# FIFA 15–22 Complete Male Player Dataset — ML Analysis
## Group: FIFA Analytics | Members: Muhammad Mudassir, Abdul Latif, Asad Ullah | Section: BsBa 6A
---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor,
    ExtraTreesRegressor, BaggingRegressor,
    RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier,
    ExtraTreesClassifier, BaggingClassifier
)
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier

print('All libraries imported successfully!')

## 2. Load & Combine FIFA 15–22 Datasets

In [ ]:
FIFA_VERSIONS = {
    'players_15.csv': 15, 'players_16.csv': 16, 'players_17.csv': 17,
    'players_18.csv': 18, 'players_19.csv': 19, 'players_20.csv': 20,
    'players_21.csv': 21, 'players_22.csv': 22
}

dfs = []
for filename, version in FIFA_VERSIONS.items():
    try:
        df_tmp = pd.read_csv(filename, low_memory=False)
        df_tmp['fifa_version'] = version
        dfs.append(df_tmp)
        print(f'Loaded {filename} — {df_tmp.shape[0]} rows')
    except FileNotFoundError:
        print(f'File not found: {filename} (skipping)')

combined = pd.concat(dfs, ignore_index=True)
print(f'\nCombined shape: {combined.shape}')

## 3. Data Preprocessing & Feature Engineering

In [ ]:
FEATURES = [
    'age', 'overall', 'potential', 'wage_eur', 'release_clause_eur',
    'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic',
    'height_cm', 'weight_kg', 'international_reputation',
    'weak_foot', 'skill_moves'
]

keep_cols = FEATURES + ['value_eur', 'player_positions', 'fifa_version',
                         'short_name', 'club_name', 'nationality_name']
available = [c for c in keep_cols if c in combined.columns]
df = combined[available].copy()

df.dropna(subset=['value_eur', 'player_positions'], inplace=True)

for col in df.select_dtypes(include='number').columns:
    df[col].fillna(df[col].median(), inplace=True)

# Feature Engineering
df['potential_gap']  = df['potential'] - df['overall']
df['wage_to_value']  = df['wage_eur'] / (df['value_eur'] + 1)
df['age_group']      = pd.cut(df['age'], bins=[15,21,27,33,50],
                               labels=['Young','Prime','Experienced','Veteran'])
le_age = LabelEncoder()
df['age_group_enc']  = le_age.fit_transform(df['age_group'])

def map_position(pos):
    pos = str(pos).upper()
    if 'GK' in pos: return 'Goalkeeper'
    for p in ['CB','LB','RB','LWB','RWB']:
        if p in pos: return 'Defender'
    for p in ['CM','CAM','CDM','LM','RM']:
        if p in pos: return 'Midfielder'
    for p in ['ST','CF','LW','RW']:
        if p in pos: return 'Attacker'
    return 'Midfielder'

df['position_cat'] = df['player_positions'].apply(map_position)

FEATURES_EXT = FEATURES + ['potential_gap','wage_to_value','age_group_enc','fifa_version']
print('Final shape:', df.shape)
print('Position distribution:\n', df['position_cat'].value_counts())

## 4. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('FIFA 15-22 Combined Data — EDA', fontsize=16, fontweight='bold')

axes[0,0].hist(np.log1p(df['value_eur']), bins=60, color='steelblue', edgecolor='white')
axes[0,0].set_title('Market Value Distribution (log scale)')
axes[0,0].set_xlabel('log(Value EUR)')

pos_counts = df['position_cat'].value_counts()
axes[0,1].bar(pos_counts.index, pos_counts.values, color=['gold','lightgreen','tomato','skyblue'])
axes[0,1].set_title('Position Distribution')

vers_avg = df.groupby('fifa_version')['overall'].mean()
axes[0,2].plot(vers_avg.index, vers_avg.values, marker='o', color='coral', linewidth=2)
axes[0,2].set_title('Avg Overall Rating by FIFA Version')
axes[0,2].set_xlabel('FIFA Version')

axes[1,0].scatter(df['overall'], np.log1p(df['value_eur']), alpha=0.02, color='purple', s=5)
axes[1,0].set_title('Overall vs Market Value')
axes[1,0].set_xlabel('Overall')

val_trend = df.groupby('fifa_version')['value_eur'].mean() / 1e6
axes[1,1].bar(val_trend.index, val_trend.values, color='steelblue', alpha=0.85)
axes[1,1].set_title('Avg Market Value by FIFA Version (M EUR)')
axes[1,1].set_xlabel('FIFA Version')

corr_cols = ['overall','potential','age','wage_eur','pace','shooting','passing','value_eur']
corr = df[[c for c in corr_cols if c in df.columns]].corr()
im = axes[1,2].imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
axes[1,2].set_xticks(range(len(corr.columns)))
axes[1,2].set_yticks(range(len(corr.columns)))
axes[1,2].set_xticklabels([c[:7] for c in corr.columns], rotation=45)
axes[1,2].set_yticklabels([c[:7] for c in corr.columns])
axes[1,2].set_title('Correlation Heatmap')
plt.colorbar(im, ax=axes[1,2])

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Messi vs Ronaldo historical comparison
name_col = 'short_name' if 'short_name' in df.columns else 'long_name'
messi   = df[df[name_col].str.contains('Messi',   case=False, na=False)]
ronaldo = df[df[name_col].str.contains('Ronaldo', case=False, na=False)]
ronaldo = ronaldo[~ronaldo[name_col].str.contains('Ronaldo Lu', case=False, na=False)]

compare_attrs = ['overall','pace','shooting','passing','dribbling','physic']
compare_attrs = [a for a in compare_attrs if a in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Messi vs Ronaldo — FIFA 15 to 22', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, attr in enumerate(compare_attrs):
    m_trend = messi.groupby('fifa_version')[attr].mean()
    r_trend = ronaldo.groupby('fifa_version')[attr].mean()
    axes[i].plot(m_trend.index, m_trend.values, color='blue', marker='o', label='Messi', linewidth=2)
    axes[i].plot(r_trend.index, r_trend.values, color='red',  marker='s', label='Ronaldo', linewidth=2)
    axes[i].set_title(attr.capitalize(), fontweight='bold')
    axes[i].set_xlabel('FIFA Version')
    if i == 0: axes[i].legend()

plt.tight_layout()
plt.savefig('messi_vs_ronaldo.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Regression — Player Market Value Prediction

In [ ]:
feat = [f for f in FEATURES_EXT if f in df.columns]
X_reg = df[feat]
y_reg = np.log1p(df['value_eur'])

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

sc_r = StandardScaler()
X_tr_sc = sc_r.fit_transform(X_train_r)
X_te_sc = sc_r.transform(X_test_r)

print(f'Train: {X_train_r.shape} | Test: {X_test_r.shape}')

In [ ]:
reg_models = {
    'Linear Regression': (LinearRegression(), True),
    'Ridge':             (Ridge(), True),
    'Lasso':             (Lasso(alpha=0.01), True),
    'ElasticNet':        (ElasticNet(alpha=0.01), True),
    'Decision Tree':     (DecisionTreeRegressor(max_depth=10, random_state=42), False),
    'KNN Regressor':     (KNeighborsRegressor(n_neighbors=5), True),
    'SVR':               (SVR(C=10), True),
    'Random Forest':     (RandomForestRegressor(n_estimators=100, random_state=42), False),
    'Extra Trees':       (ExtraTreesRegressor(n_estimators=100, random_state=42), False),
    'Bagging':           (BaggingRegressor(n_estimators=50, random_state=42), False),
    'Gradient Boosting': (GradientBoostingRegressor(n_estimators=100, random_state=42), False),
    'AdaBoost':          (AdaBoostRegressor(n_estimators=100, random_state=42), False),
    'XGBoost':           (xgb.XGBRegressor(n_estimators=200, random_state=42, verbosity=0), False),
    'LightGBM':          (lgb.LGBMRegressor(n_estimators=200, random_state=42, verbose=-1), False),
}

reg_results = []
for name, (model, scaled) in reg_models.items():
    Xtr = X_tr_sc if scaled else X_train_r
    Xte = X_te_sc if scaled else X_test_r
    model.fit(Xtr, y_train_r)
    yp  = model.predict(Xte)
    rmse = np.sqrt(mean_squared_error(y_test_r, yp))
    mae  = mean_absolute_error(y_test_r, yp)
    r2   = r2_score(y_test_r, yp)
    reg_results.append({'Model': name, 'RMSE': round(rmse,4),
                         'MAE': round(mae,4), 'R2 Score': round(r2,4)})
    print(f'{name:<22} | RMSE:{rmse:.4f} | MAE:{mae:.4f} | R2:{r2:.4f}')

reg_df = pd.DataFrame(reg_results).sort_values('R2 Score', ascending=False)
print('\nBest Model:', reg_df.iloc[0]['Model'], '| R2:', reg_df.iloc[0]['R2 Score'])

In [ ]:
# Regression comparison chart
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Regression Model Comparison — FIFA 15-22 Combined', fontsize=14, fontweight='bold')

best_r = reg_df.iloc[0]['Model']
for ax, metric, ascending in zip(axes, ['R2 Score','RMSE','MAE'], [False,True,True]):
    sorted_df = reg_df.sort_values(metric, ascending=ascending)
    best_m    = sorted_df.iloc[0]['Model']
    colors    = ['green' if m == best_m else 'steelblue' for m in sorted_df['Model']]
    ax.barh(sorted_df['Model'], sorted_df[metric], color=colors)
    ax.set_title(f'{metric} (Best = green)')

plt.tight_layout()
plt.savefig('regression_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance — XGBoost Regression
xgb_reg = xgb.XGBRegressor(n_estimators=200, random_state=42, verbosity=0)
xgb_reg.fit(X_train_r, y_train_r)
imp_r = pd.Series(xgb_reg.feature_importances_, index=feat).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
imp_r.plot(kind='barh', color='steelblue')
plt.title('Feature Importances — XGBoost Regressor', fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('reg_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Classification — Player Position Prediction

In [ ]:
le_pos = LabelEncoder()
y_cls  = le_pos.fit_transform(df['position_cat'])
X_cls  = df[feat]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

sc_c = StandardScaler()
X_tr_sc_c = sc_c.fit_transform(X_train_c)
X_te_sc_c = sc_c.transform(X_test_c)

print('Classes:', le_pos.classes_)
print(f'Train: {X_train_c.shape} | Test: {X_test_c.shape}')

In [ ]:
cls_models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
    'Decision Tree':       (DecisionTreeClassifier(max_depth=10, random_state=42), False),
    'KNN Classifier':      (KNeighborsClassifier(n_neighbors=5), True),
    'Naive Bayes':         (GaussianNB(), True),
    'SVM (RBF)':           (SVC(C=10, random_state=42), True),
    'Random Forest':       (RandomForestClassifier(n_estimators=100, random_state=42), False),
    'Extra Trees':         (ExtraTreesClassifier(n_estimators=100, random_state=42), False),
    'Bagging':             (BaggingClassifier(n_estimators=50, random_state=42), False),
    'Gradient Boosting':   (GradientBoostingClassifier(n_estimators=100, random_state=42), False),
    'AdaBoost':            (AdaBoostClassifier(n_estimators=100, random_state=42, algorithm='SAMME'), False),
    'XGBoost':             (XGBClassifier(n_estimators=200, random_state=42,
                                          use_label_encoder=False,
                                          eval_metric='mlogloss', verbosity=0), False),
    'LightGBM':            (LGBMClassifier(n_estimators=200, random_state=42, verbose=-1), False),
}

cls_results = []
for name, (model, scaled) in cls_models.items():
    Xtr = X_tr_sc_c if scaled else X_train_c
    Xte = X_te_sc_c if scaled else X_test_c
    model.fit(Xtr, y_train_c)
    yp  = model.predict(Xte)
    acc = accuracy_score(y_test_c, yp)
    f1m = f1_score(y_test_c, yp, average='macro')
    f1w = f1_score(y_test_c, yp, average='weighted')
    cls_results.append({'Model': name, 'Accuracy': round(acc,4),
                         'F1 Macro': round(f1m,4), 'F1 Weighted': round(f1w,4)})
    print(f'{name:<22} | Acc:{acc:.4f} | F1-Macro:{f1m:.4f} | F1-W:{f1w:.4f}')

cls_df = pd.DataFrame(cls_results).sort_values('Accuracy', ascending=False)
print('\nBest Model:', cls_df.iloc[0]['Model'], '| Acc:', cls_df.iloc[0]['Accuracy'])

In [ ]:
# Confusion matrix — best classifier
best_cls = XGBClassifier(n_estimators=200, random_state=42,
                          use_label_encoder=False, eval_metric='mlogloss', verbosity=0)
best_cls.fit(X_train_c, y_train_c)
yp_best = best_cls.predict(X_test_c)

cm = confusion_matrix(y_test_c, yp_best)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_pos.classes_, yticklabels=le_pos.classes_)
plt.title('Confusion Matrix — XGBoost Classifier', fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(y_test_c, yp_best, target_names=le_pos.classes_))

## 7. Hyperparameter Tuning — XGBoost (Both Tasks)

In [ ]:
param_grid = {'n_estimators':[100,200], 'max_depth':[4,6,8], 'learning_rate':[0.05,0.1,0.2]}

# Regression tuning
grid_r = GridSearchCV(xgb.XGBRegressor(random_state=42, verbosity=0),
                       param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_r.fit(X_train_r, y_train_r)
print('Regression Best Params:', grid_r.best_params_)
print('Regression Best CV R2:', round(grid_r.best_score_, 4))
print('Test R2 (tuned):', round(r2_score(y_test_r, grid_r.predict(X_test_r)), 4))

In [ ]:
# Classification tuning
grid_c = GridSearchCV(
    XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss', verbosity=0),
    param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_c.fit(X_train_c, y_train_c)
print('Classification Best Params:', grid_c.best_params_)
print('Classification Best CV Acc:', round(grid_c.best_score_, 4))
print('Test Acc (tuned):', round(accuracy_score(y_test_c, grid_c.predict(X_test_c)), 4))

## 8. Final Summary & Business Recommendations

In [ ]:
print('='*60)
print('REGRESSION SUMMARY — Player Market Value Prediction')
print('='*60)
print(reg_df.to_string(index=False))

print('\n' + '='*60)
print('CLASSIFICATION SUMMARY — Player Position Prediction')
print('='*60)
print(cls_df.to_string(index=False))

print('\n' + '='*60)
print('BUSINESS RECOMMENDATIONS')
print('='*60)
print('''
1. Transfer Valuation: Use XGBoost regression to estimate fair
   market value before transfer negotiations — prevents overpaying.

2. Player Scouting: Use classification model to auto-assign
   youth players to optimal positions based on skill attributes.

3. Youth Investment: potential_gap feature identifies undervalued
   young players with high growth — ideal for budget clubs.

4. Historical Insight: Multi-year data (FIFA 15-22) shows that
   pace and dribbling have become more valuable over time — clubs
   should prioritize these attributes in recruitment.

5. Squad Building: Combine both models — predict position and
   value of each candidate to build optimal squad within budget.
''')